# Подключение Спарк

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
pd.set_option('Display.max_columns', None)

In [2]:
import pandas as pd
import numpy as np
import re
from matplotlib import pyplot as plt
#from tqdm.notebook import tqdm
pd.set_option('Display.max_columns', None)

import sys
sys.path.append('../../ss_lal_military/src')
sys.path.append('../src/')
sys.path.append('../migrant/notebooks/utilities/')

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import os
def get_spark_session(name, level):
    """
    Get spark context
    :: name - set your app name
    :: level - set max resources level
    """
    python_path = sys.executable
    kernel = python_path.split('/')[-3]
    os.environ['SPARK_MAJOR_VERSION'] = '3'
    os.environ['SPARK_HOME'] = '/usr/sdp/current/spark3-client/'
    os.environ['PYSPARK_DRIVER_PYTHON'] = python_path
    os.environ['PYSPARK_PYTHON'] = python_path
    os.environ['LD_LIBRARY_PATH'] = '/opt/python/virtualenv/jupyter/lib'
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/')
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/lib/py4j_current')
 
    # Resources Level Profiles                           #  cpu --  ram -- desc
    if level == 1: lv = ['basic',2,10,2,10,2,2,10]       #   21 --  142 -- для базовых запросов (show create table tbl, show partitions tbl)
    if level == 2: lv = ['basic+CPU',2,10,2,10,2,2,20]   #   41 --  262 -- для простой аналитики (select * from limit 100, sum/count/avg)
    if level == 3: lv = ['middle',4,28,6,28,6,4,20]      #   81 --  742 -- для агрегатов за период 1-2мес (client_aggr_mnth, epk_campaign_daily)
    if level == 4: lv = ['middle+CPU',4,28,6,28,6,4,25]  #  101 --  912 -- для агрегатов за период >1-6мес  (client_aggr_mnth, epk_campaign_daily)
    if level == 5: lv = ['high',4,28,6,36,8,6,30]        #  121 -- 1100 -- для детальных таблиц с большими партициями (_sbol, _card, _eps)
    if level == 6: lv = ['high+CPU',4,18,5,36,8,6,40]    #  161 -- 1000 -- для детальных таблиц с мелкими партициями (feedbacks)
    if level == 7: lv = ['unfriendly',5,28,6,44,10,8,40] #  201 -- 1458 -- для запуска вечером/ночью или на пустом кластере (не рекомендуется)
    lvname = f'{level}.{lv[0]}({lv[1]*lv[7]+1},{lv[4]+lv[2]*lv[7]})'
    print(f'Kernel: {kernel}, Python_path: {python_path}, Resource_level: {lvname}')
    
    # Spark Config      
    from pyspark import SparkContext, SparkConf
    from pyspark.sql import SparkSession
  
    conf = SparkConf().setAppName(f'{name} \n ::{kernel}::{lvname}::')\
        .setMaster("yarn")\
        .set('spark.executor.cores',                     f'{lv[1]}')\
        .set('spark.executor.memory',                    f'{lv[2]}g')\
        .set('spark.executor.memoryOverhead',            f'{lv[3]}g')\
        .set('spark.driver.memory',                      f'{lv[4]}g')\
        .set('spark.driver.memoryOverhead',              f'{lv[5]}g')\
        .set('spark.driver.maxResultSize', '10g')\
        .set('spark.dynamicAllocation.initialExecutors', f'{lv[6]}')\
        .set('spark.dynamicAllocation.maxExecutors',     f'{lv[7]}')\
        .set('spark.dynamicAllocation.enabled', 'true')\
        .set('spark.dynamicAllocation.executorIdleTimeout', '120s')\
        .set('spark.dynamicAllocation.cachedExecutorIdleTimeout', '600s')\
        .set('spark.hive.mapred.supports.subdirectories', 'true')\
        .set('spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive', 'true')\
        .set('spark.shuffle.service.enabled', 'true')\
        .set('spark.port.maxRetries', '150')\
       .set('spark.sql.parquet.writeLegacyFormat', 'true')\
        .set('spark.kerberos.access.hadoopFileSystems','hdfs://arnsdpsbx:8020/')\
        .set('spark.sql.autoBroadcastJoinThreshold','20971520')
    
    spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
    return spark

try: spark
except NameError: print('Spark3 Starting')
else:
    print('Spark3 Restarting')
    spark.stop()
    
spark = get_spark_session('platon_agent_full_script_new_node', 4) # For example, MyPySpark
  
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pyspark.sql.types as T
  
sc = spark.sparkContext
sc.setLogLevel('OFF')  # or 'INFO' or 'WARN' or 'OFF'
spark


Spark3 Starting
Kernel: mlpy3811v23, Python_path: /data/sdp/mlpy3811v23/bin/python, Resource_level: 4.middle+CPU(101,728)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 10:20:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/12 10:20:29 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/05/12 10:20:29 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/05/12 10:21:06 WARN HiveConf: HiveConf of name hive.mapred.supports.subdirectories does not exist
26/05/12 10:21:07 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/05/12 10:21:19 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


# Ставим даты для бесплатного и платного уровня

In [7]:
report_dt_free = '2026-04-30'
report_dt_paid = '2026-04-30'
date_now_free = report_dt_free
date_now_paid = report_dt_paid

# Витрины для скрипта

In [5]:
aum = 'prx_bpm_stocks_custom_cib_pkaptdul.dfa_aum_clients'
ostatki_na_schetax_yul = 'prx_ostatki_na_schetax_yul_custom_cib_p4d_passive_ul.ul_balance'
aggr = 'prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth'
feedback_table = 'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_card_transactions'
pos_p2p = 'prx_bpm_pos_operations_mcc_custom_rb_card.txn_mnth_union'
romashka_hist = 'prx_bpm_model_romashka_custom_rozn_coreml_ext_scores.dm_romashka_trnsf_model_scores_vsi1_hist'
pos = 'prx_bpm_pos_operations_mcc_custom_rb_card.ft_txn_det_union'
stick = 'prx_bpm_pos_operations_mcc_custom_rb_card.scd_card_union'
potential_benefit = 'prx_bpm_potentinal_benefit_custom_rozn_profits.ft_potential_client_benefit_monthly'
sber_prime = 'prx_bpm_sber_prime_custom_rozn_sberprime.ft_sberprime'
deposit = 'prx_bpm_balances_fl_custom_rozn_cod3d_balances.cod_balances'
coins = 'prx_bpm_coins_custom_rozn_cod3d_common_coins.coins_oper'
aquaring = 'prx_bpm_transactions_other_bank_custom_b2c_acquiring.t_fct_transaction_acquiring'
tags = 'prx_bpm_tags_custom_rozn_tag.ft_master_hshtg'

# Скоринг модели платности 

In [8]:
from catboost import CatBoostClassifier
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [9]:
import logging
logging.basicConfig(level=logging.INFO, format='[%(asctime)s %(levelname)s - %(message)s]')

def normalize_column_names(data):
    logging.info("Приведение колонок к нижнему регистру")
    rename_map = {col: col.lower() for col in data.columns if col != col.lower()}
    return data.rename(columns=rename_map)


def process_date_columns(data, report_col):
    logging.info("Преобразование datetime колонок в 'дни до report_dt'")
    if report_col in data.columns:
        data[report_col] = pd.to_datetime(data[report_col].astype(str), errors="coerce")
    date_cols = [col for col in data.columns if "_dt" in col != report_col]
    for col in tqdm(date_cols, desc="Обработка дат"):
        data[col] = pd.to_datetime(data[col].astype(str), errors="coerce")
        data[col] = (data[report_col] - data[col]).dt.days
    return data.drop(columns=[report_col], errors="ignore")
    # return data

def fill_missing_values(data):
    EXCLUDED_COLS = ["report_dt"]
    logging.info("Заполнение пропусков")
    fill_config = {
        "flg":               {"fill": -1,   "type": "int32"},
        "_qty":              {"fill": -999999},
        "_pct":              {"fill": -999999},
        "_days":             {"fill": -999999},
        "_num":              {"fill": -999999},
        "_amt":              {"fill": -999999},
        "_prc":              {"fill": -999999},
        "_rate":             {"fill": -999999,   "type": "float64"},
        "_frac":             {"fill": -999999,   "type": "float64"},
        "_nflag":            {"fill": -999999,   "type": "float64"},
        "_dt":               {"fill": -999999,   "type": "float64"},
        "float64":           {"fill": -999999},
        "int32":             {"fill": -999999},
        "object":            {"fill": 'UNKNOWN'}
    }
    
    for col in tqdm(data.columns, desc="Заполнение пропусков"):
        if col in EXCLUDED_COLS:
            continue
        col_handled = False
        
        for pattern, rule in fill_config.items():
            if pattern in col:
                fill_val = rule["fill"]
                dtype = rule.get("type", None)
                data[col] = data[col].fillna(fill_val)
                if dtype:
                    data[col] = data[col].astype(dtype)
                col_handled = True
                break
                
        if not col_handled:
            col_type = str(data[col].dtype)
            if col_type in fill_config:
                fill_val = fill_config[col_type]["fill"]
                dtype = fill_config[col_type].get("type", None)
                data[col] = data[col].fillna(fill_val)
                if dtype:
                    data[col] = data[col].astype(dtype)
                    
    return data

def preprocess(data):
    logging.info("Старт предобработки")
    data = normalize_column_names(data)
    data = process_date_columns(data, report_col="report_dt")
    data = fill_missing_values(data)
    cat_features = data.select_dtypes(include="object").columns.tolist()
    logging.info(f"предобработка завершена. Число признаков: {data.shape[1]-1}")
    return data, cat_features

def prepare_dataset(train_df):
    logging.info("=== Запуск подготовки датасетов ===")
    train_df, cat_features = preprocess(train_df)
    logging.info(f"=== Подготовка завершена. Число признаков: {train_df.shape[1]-1} ===")
    return train_df, cat_features

def fix_spark_types(df):
    for col, dtype in df.dtypes:
            if "decimal" in dtype:
                df = df.withColumn(col, F.col(col).cast(T.DoubleType()))

    date_columns = [i for i in df.columns if '_dt' in i]
    for col in date_columns:
        df = df.withColumn(col, F.when(F.col(col) > F.to_date(F.lit(pd.Timestamp.max)), F.to_date(F.lit(pd.Timestamp.max))).otherwise(F.col(col)))
    
    return df

In [10]:
model_name = 'bpm_agent_paid_v2'
output_scheme = 'arnsdpsbx_team_ss' 

In [11]:
tables_dict = {'agg':'prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth',
               #'feedbacks':'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_feedbacks',
               #'vsp_visits':'prx_bpm_visiting_vsp_custom_rozn_sscxdata_cxdm.cxdm_visiting_vsp_v2',
               #'card_transactions': 'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_card_transactions',
               #'e_cod':'prx_bpm_cod_platform_cod.cod_deposit_deposit',
               #'idoc': 'prx_bpm_arrests_internal_aiv_deposit.idoc',
               #'idoc_acc': 'prx_bpm_arrests_internal_aiv_deposit.idoc_acc',
               #'pos_embeddings_fl': 'prx_bpm_pos_emb_custom_rozn_ml360.u_fl_transaction_embeddings' ,
               #'embeddings_fl': 'prx_bpm_multimodel_emb_custom_fin_palm_ml.cmn_multimodal_emb_ind_fct',
              #'card_transactions_10d': 'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_card_transactions_10d'
              }



model = CatBoostClassifier()
model.load_model('../models/final_model_new.cbm')
model_features_name = model.feature_names_

full_agg_columns = spark.read.table(tables_dict['agg']).columns
#full_emb_columns = spark.read.table(tables_dict['pos_embeddings_fl']).columns
#full_ct_columns = [i.lower() for i in spark.read.table(tables_dict['card_transactions']).columns]

features = str(model_features_name).replace('[','').replace(']','').replace("'","").replace('"','')
agg_features = str(set(model_features_name).intersection(set(full_agg_columns))).replace('{','').replace('}','').replace("'","").replace('"','')
#pos_emb_features = list(set(model_features_name).intersection(set(full_emb_columns)))
#card_transactions_features = str(set(model_features_name).intersection(set(full_ct_columns))).replace('{','').replace('}','').replace("'","").replace('"','')

In [12]:
aggr_dp_columns = []
for col in full_agg_columns:
    if (("crd" in col[0:3]) or ("dep" in col[0:3]) or ("srv" in col[0:3])):
        if (('3'not in col) 
             and ('6'not in col) 
             and ('9'not in col) 
             and ('12'not in col) 
             and ('dt'not in col) 
             and ('id'not in col) 
             and ('nflag'not in col)
             and ('cd'not in col)
             and ('dk'not in col)
             and ('rate' not in col)
             and ('1st' not in col)
             and ('lst' not in col)):
            aggr_dp_columns.append(col)
dp_features_full  = []
for col in aggr_dp_columns:
    dp_features_full = dp_features_full + [f'{col}_sum_3m', f'{col}_sum_6m', f'{col}_sum_9m', f'{col}_sum_12m', f'{col}_sum_3_6', f'{col}_sum_3_9', 
                                           f'{col}_sum_3_12', f'{col}_sum_6_9', f'{col}_sum_6_12', f'{col}_sum_9_12']
            
querry1 = ''
querry2 = ''
for col in aggr_dp_columns:
    querry1 = querry1 + f'''coalesce({col}, 0) as {col},
                         '''
    querry2 = querry2 + f'''     
                CAST(sum(coalesce(agg_3m.{col},0)) AS double) as {col}_sum_3m,
                CAST(sum(coalesce(agg_6m.{col},0)) AS double) as {col}_sum_6m,
                CAST(sum(coalesce(agg_9m.{col},0)) AS double) as {col}_sum_9m,
                CAST(sum(coalesce(agg_12m.{col},0)) AS double) as {col}_sum_12m,
                CAST(sum(coalesce(agg_3m.{col},0)) / sum(coalesce(agg_6m.{col},0)) AS double) AS {col}_sum_3_6,
                CAST(sum(coalesce(agg_3m.{col},0)) / sum(coalesce(agg_9m.{col},0)) AS double) AS {col}_sum_3_9,
                CAST(sum(coalesce(agg_3m.{col},0)) / sum(coalesce(agg_12m.{col},0)) AS double) AS {col}_sum_3_12,
                CAST(sum(coalesce(agg_6m.{col},0)) / sum(coalesce(agg_9m.{col},0)) AS double) AS {col}_sum_6_9,
                CAST(sum(coalesce(agg_6m.{col},0)) / sum(coalesce(agg_12m.{col},0)) AS double) AS {col}_sum_6_12,
                CAST(sum(coalesce(agg_9m.{col},0)) / sum(coalesce(agg_12m.{col},0)) AS double) AS {col}_sum_9_12,
                '''
querry1_clean = ''
querry2_clean = ''
for col in set(dp_features_full).intersection(set(model.feature_names_)):
    for i in querry1.split('\n'):
        if ((col[:-8] in i) | (col[:-7] in i)| (col[:-9] in i)) & (col[:-8] not in querry2_clean) & (col[:-7] not in querry2_clean) & (col[:-9] not in querry2_clean) :
            querry1_clean = querry1_clean + i + '\n'
            
    for i in querry2.split('\n'):
        if (col in i):
            querry2_clean = querry2_clean + i + '\n'

In [13]:
base = spark.sql(f'''
SELECT
    report_dt,
    epk_id,
    last_day(add_months('{date_now_paid}', -2)) AS report_dt_3m,
    last_day(add_months('{date_now_paid}', -5)) AS report_dt_6m,
    last_day(add_months('{date_now_paid}', -8)) AS report_dt_9m,
    last_day(add_months('{date_now_paid}', -11)) AS report_dt_12m
FROM
    {tables_dict['agg']}
WHERE
    report_dt = '{date_now_paid}'
    and sd_dead_nflag = 0
    and cla_all_active_1m_nflag = 1
''')





aggr_temp = spark.sql(f'''
SELECT
    epk_id,
    report_dt,
    {agg_features}
FROM 
    {tables_dict['agg']}
WHERE
    report_dt = '{date_now_paid}'
    and sd_dead_nflag = 0
    and cla_all_active_1m_nflag = 1 
''')



agg_deep = spark.sql(f'''
    SELECT
        report_dt,
        epk_id,
        {querry1_clean[:-2]}
    FROM
        {tables_dict['agg']}
    WHERE
        report_dt BETWEEN last_day(add_months('{date_now_paid}', -11)) AND '{date_now_paid}'
''')



base.createOrReplaceTempView('base')
aggr_temp.createOrReplaceTempView('aggr_temp')
agg_deep.createOrReplaceTempView('agg_deep')

agg_dynamic_features = spark.sql(f'''
    SELECT
        base.report_dt,
        base.epk_id,
        {querry2_clean[:-2]},
        avg(agg_12m.crd_otf_total_qty) as crd_otf_total_qty_12m
    FROM
        base
        LEFT JOIN agg_deep agg_3m USING (epk_id)
        LEFT JOIN agg_deep agg_6m USING (epk_id)
        LEFT JOIN agg_deep agg_9m USING (epk_id)
        LEFT JOIN agg_deep agg_12m USING (epk_id)
    WHERE
        agg_3m.report_dt BETWEEN base.report_dt_3m AND base.report_dt
        AND agg_6m.report_dt BETWEEN base.report_dt_6m AND base.report_dt
        AND agg_9m.report_dt BETWEEN base.report_dt_9m AND base.report_dt
        AND agg_12m.report_dt BETWEEN base.report_dt_12m AND base.report_dt
    GROUP BY
        base.report_dt,
        base.epk_id
''')

agg_dynamic_features.createOrReplaceTempView('agg_dynamic_features')





all_features = spark.sql(f'''
SELECT DISTINCT
    epk_id,
    report_dt,
    {features}

FROM
    base
    LEFT JOIN aggr_temp using(report_dt, epk_id)
    LEFT JOIN agg_dynamic_features using(report_dt, epk_id)
''')

all_features_fix = fix_spark_types(all_features)
all_features_fix.write.saveAsTable(f"{output_scheme}.{model_name}_features_scoring", mode = 'overwrite')

In [14]:
model = CatBoostClassifier()
model.load_model('../models/final_model_new.cbm')
feats = model.feature_names_
model = spark.sparkContext.broadcast(model)
def func_(frames):
    clf = model.value
    for frame in frames:
        report_dt_col = frame['report_dt'].copy()
        epk_id_col = frame['epk_id'].copy()
        frame, _ = preprocess(frame)
        pred = clf.predict_proba(frame[feats])
        frame['score'] = pred[:, 1]
        frame['report_dt'] = report_dt_col.values
        frame = frame[['epk_id', 'score', 'report_dt']]
        frame.epk_id = frame.epk_id.astype(int)
        yield frame

        
        
        
        
spark_df = spark.read.table(f"{output_scheme}.{model_name}_features_scoring")

from pyspark.sql.functions import *
import pyspark.sql.types as T
out_schema = StructType(
    fields = [
        T.StructField('epk_id', T.LongType(), True),
        T.StructField('score', T.FloatType(), True),
        T.StructField('report_dt', T.StringType(), True)])

result = spark_df.mapInPandas(func_, out_schema)

spark.sql(f'''drop table {output_scheme}.{model_name}_scores''')

new_table = False
try: 
    spark.read.table(f'{output_scheme}.{model_name}_scores')
except: 
    new_table = True


if new_table:
    result.write.saveAsTable(f"{output_scheme}.{model_name}_scores", mode = 'overwrite')
else:
    if date_now_paid in list(spark.read.table(f'{output_scheme}.{model_name}_scores').select('report_dt').distinct().toPandas().report_dt.astype(str)):
        result_union = spark.read.table(f'{output_scheme}.{model_name}_scores').filter(F.col('report_dt') != date_now_paid)
        result_union = result_union.union(result)
        result_union.write.saveAsTable(f"{output_scheme}.{model_name}_scores_", mode = 'overwrite')
        result_union = spark.read.table(f'{output_scheme}.{model_name}_scores_')
        result_union.write.saveAsTable(f"{output_scheme}.{model_name}_scores", mode = 'overwrite')
        spark.sql(f'drop table {output_scheme}.{model_name}_scores_')
    else:
        result.write.saveAsTable(f"{output_scheme}.{model_name}_scores", mode = 'append')   

In [18]:
spark.sql(f'''
select count(epk_id), count(distinct epk_id) from arnsdpsbx_team_ss.bpm_agent_paid_v2_scores 
''').show()

+-------------+----------------------+
|count(epk_id)|count(DISTINCT epk_id)|
+-------------+----------------------+
|    106861606|             106861606|
+-------------+----------------------+

